In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from stoneforge.data_management.preprocessing import DataLoader, DataManager

# Manual Access:
las2 = DataLoader(r"https://raw.githubusercontent.com/giecaruff/datasets/refs/heads/main/wells/las2/npra/IK1.las", filetype='las2')
print('header itens:',las2.data_obj.header.keys())
las2.data_obj.header['well']

header itens: dict_keys(['version', 'well', 'curve', 'parameter', 'other'])


,mnemonic,unit,value,description
0,STRT,F,81.0000,START DEPTH
1,STOP,F,15400.0000,STOP DEPTH
2,STEP,F,0.5000,STEP VALUE
3,NULL,,-999.2500,NULL VALUE
4,COMP,,USGS/NPR HUSKY OIL OPERAT,COMPANY
5,WELL,,IKPIKPUK TEST WELL #1,WELL
6,FLD,,WILDCAT,FIELD
7,LOC,,25 13N 10W,LOCATION
8,CNTY,,NORTH SLPOE,COUNTY
9,STAT,,ALASKA,STATE


In [4]:
# Example (Manual Access): Accessing data as DataFrame
data_las2, units_las2 = las2.dataframe(las2.data_obj.data)
data_las2 = data_las2.replace(-999.0, np.nan)
data_las2

,DEPT,SP,ILD,ILM,LL8,GR,CALI,RHOB,DRHO,NPHI,DT
0,81.0,NaN,NaN,NaN,NaN,79.7502,NaN,NaN,NaN,NaN,NaN
1,81.5,NaN,NaN,NaN,NaN,79.9790,NaN,NaN,NaN,NaN,NaN
2,82.0,NaN,NaN,NaN,NaN,79.8643,NaN,NaN,NaN,NaN,NaN
3,82.5,NaN,NaN,NaN,NaN,79.9446,NaN,NaN,NaN,NaN,NaN
4,83.0,NaN,NaN,NaN,NaN,80.1459,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
30796,15479.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30797,15479.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30798,15480.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30799,15480.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Adding facies
IK1 = DataManager(las2, depth="DEPT")

IK1.add_facie(name="LEDGE_SANDSTONE", top=10619, bottom=10842)

# View Facies LEDGE_SANDSTONE interval
IK1.LEDGE_SANDSTONE

,DEPT,SP,ILD,ILM,LL8,GR,CALI,RHOB,DRHO,NPHI,DT
21076,10619.0,-92.3719,5.6534,7.5047,10.3189,32.1681,9.5756,2.4324,0.0280,18.5964,74.0643
21077,10619.5,-92.6030,5.4512,7.1849,9.8774,29.9718,9.5682,2.4169,0.0243,18.5066,73.7811
21078,10620.0,-92.8108,5.2563,6.9832,9.3140,28.8898,9.5609,2.4278,0.0214,18.2585,73.3278
21079,10620.5,-93.0186,5.0684,6.7872,8.3432,28.8571,9.5536,2.4187,0.0232,18.0104,72.9256
21080,10621.0,-93.2263,4.9828,6.7115,7.1612,29.6683,9.5463,2.3921,0.0221,17.8458,73.9007
...,...,...,...,...,...,...,...,...,...,...,...
21518,10840.0,-84.3232,15.8031,24.9516,38.3127,38.1211,9.6802,2.5684,0.0246,9.2279,65.8097
21519,10840.5,-83.6495,18.1204,25.9377,29.7650,36.1407,9.6372,2.5561,0.0323,8.8146,63.3392
21520,10841.0,-82.9757,23.8513,37.9959,22.8938,36.0931,9.5941,2.5345,0.0364,8.6617,63.0884
21521,10841.5,-82.3019,24.9165,54.6157,25.3613,37.6945,9.5790,2.5176,0.0275,8.5357,63.5907


In [6]:
# Solução de sistema (teste simplificado)

DEPT = np.array(IK1.LEDGE_SANDSTONE['DEPT'])
GR = np.array(IK1.LEDGE_SANDSTONE['GR'])
DT = np.array(IK1.LEDGE_SANDSTONE['DT'])
NPHI = np.array(IK1.LEDGE_SANDSTONE['NPHI'])
RHOB = np.array(IK1.LEDGE_SANDSTONE['RHOB'])

# A0 = Matriz de valores tabelados

A0 = np.array(
    [
        [20,11,111,160,0.0001], # GR
        [2.650,2.710,2.657,2.56,1.100], # RHOB
        [0.000,0.000,48.1,40.0,100.0], # NPHI
        [55.5,47.8,100,130,185], # DT
        [1.0,1.0,1.0,1.0,1.0] # 1
    ]
)

# Demais operações:

AI0 = np.linalg.inv(A0)

X0 = []
for i in range (len(GR)):
    B0 = np.array([GR[i],RHOB[i],NPHI[i],DT[i],1.0],float)
    x = np.dot(AI0,B0)
    if i == 0:
        print("Valores encontrados:",B0)
        print("Proporção da composição:",x)
    X0.append(x)
X0 = np.array(X0)

Valores encontrados: [32.1681  2.4324 18.5964 74.0643  1.    ]
Proporção da composição: [ 5.34221072 -4.54931849  1.10481576 -0.92042601  0.02271803]


In [7]:
# A1 = Matriz de valores tabelados

A1 = np.array(
    [
    [20,11,111,160], # GR
    [2.650,2.710,2.657,2.56], # RHOB
    [0.000,0.000,48.1,40.0], # NPHI
    [55.5,47.8,100,130], # DT
    [1.0,1.0,1.0,1.0] # 1
    ]
    )

# Demais operações:

AA1 = np.dot(A1.T,A1)
AI1 = np.linalg.inv(AA1)
AT1 = np.dot(A1,AI1)

X1 = []
for i in range (len(GR)):
    B1 = np.array([GR[i],RHOB[i],NPHI[i],DT[i],1.0],float)
    x = np.dot(B1,AT1)
    if i == 0:
        print("Valores encontrados:",B1)
        print("Proporção da composição:",x)
    X1.append(x)
X1 = np.array(X1)

Valores encontrados: [32.1681  2.4324 18.5964 74.0643  1.    ]
Proporção da composição: [ 6.71829622 -5.8985014   1.37223449 -1.18520199]


In [8]:
# A2 = Matriz de valores tabelados

A2 = np.array([
    #[20,11,111,160,0.0001], # GR
    [2.650,2.710,2.657,2.56,1.10], # RHOB
    [0.000,0.000,48.1,40.0,100.00], # NPHI
    [55.5,47.8,100,130,185], # DT
    [1.0,1.0,1.0,1.0,1.0] # 1
])

# Demais operações:

AA2 = np.dot(A2,A2.T)
AI2 = np.linalg.inv(AA2)
AT2 = np.dot(AI2,A2)

X2 = []
for i in range (len(GR)):
    B2 = np.array([RHOB[i],NPHI[i],DT[i],1.0],float)
    x = np.dot(B2,AT2)
    if i == 0:
        print("Valores encontrados:",B2)
        print("Proporção da composição:",x)
    X2.append(x)
X2 = np.array(X2)

Valores encontrados: [ 2.4324 18.5964 74.0643  1.    ]
Proporção da composição: [ 0.38519199  0.40556177  0.0875366  -0.03691542  0.15862506]


In [9]:
class Elan:

    def __init__(self,md,gr,rhob,nphi,dt,lito = False):
        self.md = md
        self.lito = lito #if lito.any:
        self.m = len(md)

        self.litho_code = {
            57:"green",
            49:"yellow",
            54:"maroon",
            25:"grey"
            }

        ones_log = np.ones(self.m).T

        # valores na ordem: (quartzo, calcita, lama, arcóseo, fluido)
        gr_values = np.array([20,11,111,160,0.0001])
        dt_values = np.array([55.5,47.8,100,130,185])
        rhob_values = np.array([2.650,2.710,2.657,2.56,1.100])
        nphi_values = np.array([0.000,0.000,48.1,40.0,100.0])
        ones_values = np.array([1.0,1.0,1.0,1.0,1.0])

        self.matrix = np.array([gr_values,dt_values,rhob_values,nphi_values,ones_values])
        self.elems = {'qtz':0,'cal':1,'shl':2,'ark':3,'fld':4}
        self.mnems = {'gr':0,'dt':1,'rhob':2,'nphi':3,'ones':4}
        self.datst = np.array([gr,dt,rhob,nphi,ones_log],float)
        self.names = ['quartzo','calcita','lama','arcóseo','fluido']
        self.elem_colors = ['#eaec61','#6fb5db','#438d8e','orange','navy']
        self.mnem_colors = ['green','black','red','blue']
        self.mnems_names = ['GR','DT','RHOB','NPHI']
        self.mnesm_units = ['API','us/ft','g/cm3','v/v']

    def matrix_crop(self,mnems = [],elems = []):

        self.mnems_val = [self.mnems[x] for x in mnems]
        self.elems_val = [self.elems[x] for x in elems]

        set5 = set(range(5))

        mnems_list = set5 - set(set5 - set(self.mnems_val))
        elems_list = set5 - set(set5 - set(self.elems_val))

        matrix = self.matrix
        submatrix = matrix[np.ix_(self.mnems_val,self.elems_val)]

        datst = self.datst[np.ix_(self.mnems_val)]

        return (submatrix,mnems_list,elems_list,datst)

    # ======================================================================#


    def system_sol(self, info, reg = 0.0, min_r = False):

        A0 = info[0]
        datst = info[3]
        self.ssol_mnems = info[1]
        self.ssol_elems = info[2]

        X0 = []
        AI0 = np.linalg.inv(A0 + (np.eye(A0.shape[0])*reg))

        for i in range (self.m):
            B0 = datst[:,i]
            x = np.dot(AI0,B0)
            X0.append(x)
        X0 = np.array(X0)

        if min_r:
            X0 = self._min_r(X0)

        self.ssol_x0 = X0

    def min_qd(self, info, reg = 0.0, min_r = False):

        A1 = info[0]
        datst = info[3]
        self.mnqd_mnems = info[1]
        self.mnqd_elems = info[2]

        AA1 = np.dot(A1.T,A1)
        AI1 = np.linalg.inv(AA1 + (np.eye(AA1.shape[0])*reg))
        AT1 = np.dot(A1,AI1)

        X1 = []
        for i in range (self.m):
            B1 = datst[:,i]
            x = np.dot(B1,AT1)
            X1.append(x)
        X1 = np.array(X1)

        if min_r:
            X1 = self._min_r(X1)

        self.mnqd_x1 = X1

    def moore_pen(self, info, reg = 0.0, min_r = False):

        A2 = info[0]
        datst = info[3]
        self.mrpen_mnems = info[1]
        self.mrpen_elems = info[2]

        AA2 = np.dot(A2,A2.T)
        AI2 = np.linalg.inv(AA2 + (np.eye(AA2.shape[0])*reg))
        AT2 = np.dot(AI2,A2)

        X2 = []
        for i in range (len(GR)):
            B2 = datst[:,i]
            x = np.dot(B2,AT2)
            X2.append(x)
        X2 = np.array(X2)

        if min_r:
            X2 = self._min_r(X2)

        self.mrpen_x2 = X2

    # ======================================================================#

    def _min_r(self,X0):

        m,n = np.shape(X0)
        aux = np.copy(X0)
        for i in range(n):
            X0[:,i] = aux[:,i] - np.min(aux[:,i])

        return X0

In [10]:
# Elan Code:
EE = Elan(DEPT,GR,RHOB,NPHI,DT)

#A0 = EE.matrix_crop(['gr','rhob','ones'],['qtz','shl','fld']) # fld, qtz, shl cal, ark
A0 = EE.matrix_crop(['rhob','gr','ones'],['fld', 'qtz', 'shl'])

EE.system_sol(A0, reg = 0.30, min_r = True)